In [1]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV


import pandas as pd
import numpy as np

import joblib

import utils as ut

In [2]:
import os

In [3]:
# ===================== Load Data =====================
df, dict_info = ut.load_sov_dataset("risky")

In [4]:
# compute z scores across groups of columns
tf = "z"  # "z" or "mean_center_only" or "log_values"
df = ut.zscore_grouped_cols(df, dict_info["l_colnames_grouped_zscale"], tf=tf)

In [5]:
df_original, df_id_nohist, df_shared_hist, df_shared_nohist = ut.make_conditions(df)

In [6]:
fit_models = True

# Average Model No History

In [19]:
df = df_original.copy()
df["is_train"] = df["trial_id"] <= 60
dict_groups = dict(tuple(df.groupby('is_train')))
df_train = dict_groups[True]
df_dev = dict_groups[False]

In [20]:
X_train = df_train.head(100000)[dict_info["cols_x"]].copy()
y_train = np.ravel(df_train.head(100000)[dict_info["col_y"]].copy())
X_dev = df_dev.head(100000)[dict_info["cols_x"]].copy()
y_dev = np.ravel(df_dev.head(100000)[dict_info["col_y"]].copy())

In [23]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.05, 0.1, 0.15],
    "max_depth": [2, 3, 4]
}

In [ ]:
model_full = GradientBoostingClassifier()

grid_full = GridSearchCV(
    estimator=model_full,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="neg_log_loss",  # or another metric
    n_jobs=-1            # use all CPU cores
)

if fit_models:
    grid_full.fit(X_train, y_train)
    joblib.dump(grid_full, "models/risky-sklearn/gridsearch-all-vals-and-probs.pkl")
else:
    grid_full = joblib.load("models/risky-sklearn/gridsearch-all-vals-and-probs.pkl")

In [ ]:
best_model_full = grid_full.best_estimator_
y_dev_pred = best_model_full.predict(X_dev)
y_train_pred = best_model_full.predict(X_train)
df_train_preds_full = pd.DataFrame({"y_train_true":y_train, "y_train_pred":y_train_pred}) 
df_train_preds_full["is_correct"] = df_train_preds_full["y_train_true"] == df_train_preds_full["y_train_pred"]
df_dev_preds_full = pd.DataFrame({"y_dev_true":y_dev, "y_dev_pred":y_dev_pred})
df_dev_preds_full["is_correct"] = df_dev_preds_full["y_dev_true"] == df_dev_preds_full["y_dev_pred"]

In [ ]:
print(
    "ALL VALUES AND PROBABILITIES:\n",
    "train accuracy: ", np.round(df_train_preds_full["is_correct"].mean(), 3), 
    "\ndev accuracy: ", np.round(df_dev_preds_full["is_correct"].mean(), 3)
)

# Individual Differences Model No History

In [41]:
X_train = df_train.head(100000)[dict_info["cols_x"] + dict_info["col_pid"]].copy()
y_train = np.ravel(df_train.head(100000)[dict_info["col_y"]].copy())
X_dev = df_dev.head(100000)[dict_info["cols_x"] + dict_info["col_pid"]].copy()
y_dev = np.ravel(df_dev.head(100000)[dict_info["col_y"]].copy())

In [42]:
model_ID_full = GradientBoostingClassifier()

grid_ID_full = GridSearchCV(
    estimator=model_ID_full,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="neg_log_loss",  # or another metric
    n_jobs=-1            # use all CPU cores
)

if fit_models:
    grid_ID_full.fit(X_train, y_train)
    joblib.dump(grid_ID_full, "models/risky-sklearn/gridsearch-ID-all-vals-and-probs.pkl")
else:
    grid_ID_full = joblib.load("models/risky-sklearn/gridsearch-ID-all-vals-and-probs.pkl")

In [43]:
best_model_ID_full = grid_ID_full.best_estimator_
y_dev_pred = best_model_ID_full.predict(X_dev)
y_train_pred = best_model_ID_full.predict(X_train)
df_train_preds_ID_full = pd.DataFrame({"y_train_true":y_train, "y_train_pred":y_train_pred}) 
df_train_preds_ID_full["is_correct"] = df_train_preds_ID_full["y_train_true"] == df_train_preds_ID_full["y_train_pred"]
df_dev_preds_ID_full = pd.DataFrame({"y_dev_true":y_dev, "y_dev_pred":y_dev_pred})
df_dev_preds_ID_full["is_correct"] = df_dev_preds_ID_full["y_dev_true"] == df_dev_preds_ID_full["y_dev_pred"]

In [44]:
print(
    "ID: ALL VALUES AND PROBABILITIES:\n",
    "train accuracy: ", np.round(df_train_preds_ID_full["is_correct"].mean(), 3), 
    "\ndev accuracy: ", np.round(df_dev_preds_ID_full["is_correct"].mean(), 3)
)

ID: ALL VALUES AND PROBABILITIES:
 train accuracy:  0.599 
dev accuracy:  0.611
